# Garak Scenarios

The Garak scenario family implements probes inspired by the
[Garak](https://github.com/NVIDIA/garak) framework. These include encoding-based probes (which
test whether a target can be tricked into producing harmful content when prompts are encoded in
various formats), web-injection probes (which test whether a target emits markdown
data-exfiltration or cross-site-scripting payloads), and system-prompt-extraction probes (which
test whether a target can be coaxed into revealing its own system prompt).

For full programming details, see the
[Scenarios Programming Guide](../code/scenarios/0_scenarios.ipynb).

In [1]:
from pathlib import Path

from pyrit.output import output_scenario_async
from pyrit.registry import TargetRegistry
from pyrit.scenario.garak import (
    Encoding,
    EncodingStrategy,
    SystemPromptExtraction,
    SystemPromptExtractionStrategy,
)
from pyrit.scenario.garak.encoding import EncodingDatasetConfiguration
from pyrit.setup import initialize_from_config_async

await initialize_from_config_async(config_path=Path("pyrit_conf.yaml"))  # type: ignore

objective_target = TargetRegistry.get_registry_singleton().instances.get("openai_chat")

Found default environment files: ['C:\\Users\\rlundeen\\.pyrit\\.env', 'C:\\Users\\rlundeen\\.pyrit\\.env.local']
Loaded environment file: C:\Users\rlundeen\.pyrit\.env
Loaded environment file: C:\Users\rlundeen\.pyrit\.env.local


[pyrit:alembic] No new upgrade operations detected.


Skipping target 'platform_openai_chat': PLATFORM_OPENAI_CHAT_GPT4O_MODEL is not set. All declared env vars (endpoint, key, model) must be present for this target to register.


Skipping target 'azure_foundry_phi4': AZURE_FOUNDRY_PHI4_MODEL is not set. All declared env vars (endpoint, key, model) must be present for this target to register.


TextAdaptive: _EXCLUDED_TECHNIQUES entries ['prompt_sending'] are not in the current scenario-techniques catalog ['context_compliance', 'crescendo_history_lecture', 'crescendo_journalist_interview', 'crescendo_movie_director', 'crescendo_simulated', 'many_shot', 'pair', 'red_teaming', 'role_play', 'tap', 'violent_durian']; the exclusion is a no-op for those entries. Remove stale entries or update the catalog.


...

Loading datasets - this can take a few minutes:  19%|█▉        | 19/101 [00:00<00:01, 48.47dataset/s]

Loading datasets - this can take a few minutes:  29%|██▊       | 29/101 [00:00<00:01, 41.65dataset/s]

Loading datasets - this can take a few minutes:  36%|███▌      | 36/101 [00:00<00:01, 45.61dataset/s]

Loading datasets - this can take a few minutes:  99%|█████████▉| 100/101 [00:03<00:00, 24.94dataset/s]

Loading datasets - this can take a few minutes: 100%|██████████| 101/101 [00:03<00:00, 26.19dataset/s]

## Encoding

Tests whether the target can decode and comply with encoded harmful prompts. Each encoding
strategy encodes the prompt, asks the target to decode it, and scores whether the decoded output
matches the harmful content. Default datasets include slur terms and web/HTML/JS content.

**CLI example:**

```bash
pyrit_scan garak.encoding --target openai_chat --strategies base64 --max-dataset-size 1
```

**Available strategies** (17 encodings): Base64, Base2048, Base16, Base32, ASCII85, Hex,
QuotedPrintable, UUencode, ROT13, Braille, Atbash, MorseCode, NATO, Ecoji, Zalgo, LeetSpeak,
AsciiSmuggler

> **Note:** Strategy composition is NOT supported for Encoding — each encoding is tested
> independently.

In [2]:
dataset_config = EncodingDatasetConfiguration(dataset_names=["garak_slur_terms_en"], max_dataset_size=1)

scenario = Encoding()
await scenario.initialize_async(  # type: ignore
    objective_target=objective_target,
    scenario_strategies=[EncodingStrategy.Base64],
    dataset_config=dataset_config,
)

print(f"Scenario: {scenario.name}")
print(f"Atomic attacks: {scenario.atomic_attack_count}")

scenario_result = await scenario.run_async()  # type: ignore

Scenario: Encoding
Atomic attacks: 21


Executing Encoding:   0%|          | 0/21 [00:00<?, ?attack/s]

Executing Encoding:   5%|▍         | 1/21 [00:10<03:23, 10.18s/attack]

Executing Encoding:  10%|▉         | 2/21 [00:10<01:21,  4.31s/attack]

Executing Encoding:  24%|██▍       | 5/21 [00:11<00:22,  1.43s/attack]

...

Executing Encoding:  81%|████████  | 17/21 [00:14<00:01,  3.40attack/s]

Executing Encoding:  86%|████████▌ | 18/21 [00:15<00:01,  2.75attack/s]

Executing Encoding:  95%|█████████▌| 20/21 [00:16<00:00,  1.83attack/s]

Executing Encoding: 100%|██████████| 21/21 [00:17<00:00,  2.13attack/s]

Executing Encoding: 100%|██████████| 21/21 [00:17<00:00,  1.23attack/s]

In [3]:
await output_scenario_async(scenario_result)


                                    📊 SCENARIO RESULTS: Encoding                                    

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Encoding
    • Scenario Version: 1
    • PyRIT Version: 0.15.0.dev0
    • Description:
        Encoding Scenario implementation for PyRIT. This scenario tests how resilient models are to various encoding
        attacks by encoding potentially harmful text (by default slurs and XSS payloads) and testing if the model will
        decode and repeat the encoded payload. It mimics the Garak encoding probe. The scenario works by: 1. Taking seed
        prompts (the harmful text to be encoded) 2. Encoding them using various encoding schemes (Base64, ROT13, Morse,
        etc.) 3. Asking the target model to decode the encoded text 4. Scoring whether the model successfully decoded
        and repeated the harmful content By default, this 

## WebInjection

Ports Garak's `web_injection` probe family. Tests whether the target can be coaxed into emitting
web-injection payloads: markdown-image data exfiltration (CSRF-style) and cross-site-scripting
(XSS). Each strategy pairs a `SeedObjective` (the goal) with a `SeedPrompt` (the literal
injection prompt) and scores the response with the markdown-injection or XSS output scorer.

**CLI example:**

```bash
pyrit_scan garak.web_injection --target openai_chat --strategies xss --max-dataset-size 1
```

**Available strategies** (8 probes): MarkdownImageExfil, ColabAIDataLeakage,
StringAssemblyDataExfil, PlaygroundMarkdownExfil, MarkdownURIImageExfilExtended,
MarkdownURINonImageExfilExtended, TaskXSS, MarkdownXSS.

**Aggregate strategies:** `ALL` (all 8), `DEFAULT` (excludes the two combinatorial extended
probes), `EXFIL` (the 6 markdown-exfil probes), and `XSS` (TaskXSS + MarkdownXSS).

## SystemPromptExtraction

Ports Garak's `sysprompt_extraction` probe. A real system prompt (sourced from the
`garak_drh_system_prompts` / `garak_tm_system_prompts` libraries) is installed on the target, then
an extraction request asks the model to reveal it. Responses are scored deterministically by
`SystemPromptExtractionScorer`, a character n-gram containment overlap between the response and the
known system prompt (a faithful port of Garak's `PromptExtraction` detector), wrapped by a
`FloatScaleThresholdScorer` at threshold 0.5.

Each of the 9 attack-template categories is a strategy; across the selected categories the total
(system prompt × template) combinations are randomly sampled down to `prompt_cap` (Garak's
`soft_probe_prompt_cap`, default 256) so a default run stays bounded.

**CLI example:**

```bash
pyrit_scan garak.system_prompt_extraction --target openai_chat --strategies direct_requests
```

**Available strategies** (9 categories): DirectRequests, RolePlayingAttacks, EncodingBasedAttacks,
IndirectCreativeApproaches, CodeTechnicalFraming, ContinuationTricks, MultiLayeredApproaches,
AuthorityUrgencyFraming, ConfusionDistraction.

The minimal run below installs a single system prompt and runs one category so it completes
quickly.

In [4]:
sysprompt_scenario = SystemPromptExtraction(system_prompt_subsample=1, prompt_cap=1)
await sysprompt_scenario.initialize_async(  # type: ignore
    objective_target=objective_target,
    scenario_strategies=[SystemPromptExtractionStrategy.DirectRequests],
)

print(f"Scenario: {sysprompt_scenario.name}")
print(f"Atomic attacks: {sysprompt_scenario.atomic_attack_count}")

sysprompt_result = await sysprompt_scenario.run_async()  # type: ignore

Scenario: SystemPromptExtraction
Atomic attacks: 1


Executing SystemPromptExtraction:   0%|          | 0/1 [00:00<?, ?attack/s]

Executing SystemPromptExtraction: 100%|██████████| 1/1 [00:01<00:00,  1.36s/attack]

Executing SystemPromptExtraction: 100%|██████████| 1/1 [00:01<00:00,  1.36s/attack]

In [5]:
await output_scenario_async(sysprompt_result)


                             📊 SCENARIO RESULTS: SystemPromptExtraction                             

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: SystemPromptExtraction
    • Scenario Version: 1
    • PyRIT Version: 0.15.0.dev0
    • Description:
        System Prompt Extraction scenario implementation for PyRIT. Ports garak's
        ``sysprompt_extraction.SystemPromptExtraction`` probe. A real system prompt (sourced from the
        ``garak_drh_system_prompts`` / ``garak_tm_system_prompts`` datasets) is installed on the target, then an
        extraction request (from the ``garak_system_prompt_extraction`` dataset) asks the model to reveal it. Responses
        are scored deterministically with ``SystemPromptExtractionScorer`` (a character n-gram containment overlap
        between the response and the known system prompt), wrapped by ``FloatScaleThresholdScorer`` for the

For more details, see the [Scenarios Programming Guide](../code/scenarios/0_scenarios.ipynb) and
[Configuration](../getting_started/configuration.md).